# Commodity Flow Intelligence — Market Briefing

**Physical supply risk assessment from public EIA, USGS, and market data.**

This briefing synthesizes delivery gap signals, inventory adequacy, wellhead economics, and futures pricing into a single risk view. Start here, then drill into the deep-dive notebooks for evidence and root-cause analysis.

| Deep-Dive | Focus |
|-----------|-------|
| [01 — Delivery Gap Analysis](01_delivery_gap_analysis.ipynb) | Where are the physical supply gaps? |
| [02 — Wellhead Economics](02_wellhead_economics.ipynb) | Why are gaps forming? Which basins are shutting in? |

In [ ]:
# --- Colab Setup (skip if running locally) ---
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/hb-cam/commodity-flow-intelligence.git
    import os
    from getpass import getpass
    _eia_key = getpass('EIA API Key (Enter to skip): ')
    if _eia_key:
        os.environ['EIA_API_KEY'] = _eia_key
        os.environ['USE_LIVE_API'] = 'true'
    print('Setup complete.' + (' Live mode enabled.' if _eia_key else ' Using offline data.'))

#### Setup

In [ ]:
from datetime import datetime
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from commodity_flow import analysis, charts, config, inventory, offline
from commodity_flow.refresh import RefreshPipeline

# Load all data via the refresh pipeline (includes validation)
pipeline = RefreshPipeline()
data = pipeline.run()

if not pipeline.all_passed:
    print('\u26a0\ufe0f VALIDATION WARNINGS:')
    for v in pipeline.validations:
        if not v.passed:
            print(f'  {v.name}: {v.detail}')
else:
    print('\u2705 All data validation checks passed.')

# Load inventory data
if config.USE_LIVE_API and config.EIA_API_KEY:
    df_prod_stocks = inventory.fetch_product_stocks(config.EIA_API_KEY)
    df_prod_supplied = inventory.fetch_product_supplied(config.EIA_API_KEY)
else:
    inv_data = inventory.generate_offline_inventory()
    df_prod_stocks = inv_data['stocks']
    df_prod_supplied = inv_data['supplied']

# Build derived datasets
scorecard = analysis.build_scorecard(data['imports'], data['natgas'], data['steo'])
df_breakevens = data['breakevens']
current_wti = df_breakevens.sort_values('date').iloc[-1]['wti_price_usd_bbl']
status = analysis.compute_breakeven_status(df_breakevens, current_wti)
risk_curve = analysis.production_at_risk_curve(df_breakevens, data['dpr'])
df_dos = inventory.compute_days_of_supply(df_prod_stocks, df_prod_supplied)
df_spr = inventory.compute_spr_status(df_prod_stocks)
df_seasonal = inventory.compute_seasonal_comparison(df_prod_stocks)

briefing_date = datetime.now().strftime('%B %d, %Y')
print(f'\nBriefing date: {briefing_date}')
print(f'WTI reference: ${current_wti:.0f}/bbl')

---
## Risk Dashboard

In [ ]:
fig = charts.plot_risk_dashboard(scorecard, status, df_spr, df_dos, current_wti)
fig.show()

---
## Signal Status

Automated signal checks against predefined thresholds. Signals are graded:
- 🔴 **ALERT** — threshold breached, requires attention
- ⚠️ **WARNING / WATCH** — approaching threshold
- ✅ **OK** — within normal range

In [ ]:
signal_table = charts.build_signal_table(
    scorecard, data['imports'], df_dos, df_spr, status, data['dpr']
)
display(signal_table.style.set_properties(**{
    'text-align': 'left',
    'font-size': '13px'
}).hide(axis='index'))

---
## Composite Gap Scorecard

Blends crude oil import z-scores with natural gas import z-scores into a single delivery-risk metric. STEO forward overlay shows where EIA expects the signal to go.

> **Reading this chart:** Sustained readings below -1\u03c3 indicate physical supply tighter than trailing norms. Below -2\u03c3 is a critical alert. The dashed line shows STEO-implied forward path.

In [ ]:
# Compute STEO historical accuracy for confidence band
steo_accuracy = analysis.compute_steo_accuracy(data['imports'], data['steo'])
if steo_accuracy['n_comparisons'] > 0:
    print(f'STEO accuracy ({steo_accuracy["n_comparisons"]} months): '
          f'MAE={steo_accuracy["mae"]:.2f}, RMSE={steo_accuracy["rmse"]:.2f}, '
          f'\u00b11\u03c3 band={steo_accuracy["band_1sigma"]:.2f}')
else:
    print('Insufficient overlap to compute STEO accuracy.')

fig = charts.plot_scorecard(scorecard, steo_accuracy=steo_accuracy)
fig.show()

---
## Scenario Analysis — What Breaks at Each WTI Price?

The US shale marginal cost curve shows how much production is at risk of shut-in at different WTI levels. This is the supply-side elasticity that drives delivery gaps downstream.

> **Reading this chart:** Hover over any WTI price to see how many barrels/day of production are at risk. The vertical dashed line marks the current WTI.

In [ ]:
fig = charts.plot_elasticity_curve(risk_curve, current_wti)
fig.show()

# Scenario summary
scenarios = [50, current_wti, 90]
print('\n=== Scenario Summary ===')
for wti in scenarios:
    s = analysis.compute_breakeven_status(df_breakevens, wti)
    n_risk = (s['status'] == 'at risk').sum()
    at_risk_names = ', '.join(s[s['status'] == 'at risk']['basin'].tolist()) or 'None'
    row = risk_curve.iloc[(risk_curve['wti_price'] - wti).abs().argsort()[:1]]
    pct = row['pct_at_risk'].iloc[0] if not row.empty else 0
    label = 'BEAR' if wti < current_wti else ('BASE' if wti == current_wti else 'BULL')
    print(f'  {label} (${wti:.0f}): {n_risk} basins at risk ({at_risk_names}), {pct:.0f}% production at risk')

---
## Basin Profitability

Current breakeven costs by basin vs prevailing WTI. Green = profitable, yellow = marginal (within $10), red = below breakeven.

> **Reading this chart:** Basins to the right of the WTI line are losing money on new wells. Sustained unprofitability leads to rig drops, completions decline, and supply contraction within 3-6 months.

In [ ]:
fig = charts.plot_basin_breakevens(status, current_wti)
fig.show()

---
## Distillate Supply Chain

Interactive Sankey diagram tracing crude oil from source through refinery to distillate end uses. Distillate (diesel, heating oil, industrial fuel) is the product most sensitive to inventory tightness.

> **Reading this chart:** The orange flows trace distillate through the system. On-highway diesel is the largest draw. When distillate days-of-supply drops below 30, pricing pressure concentrates in these downstream channels.

In [ ]:
fig = charts.plot_distillate_sankey()
fig.show()

---
## Inventory Adequacy

### Days of Supply by Product

Stocks divided by daily consumption. Red reference line = danger threshold.

> **Reading this chart:** Below the threshold, physical supply is tight enough to move spot prices. Gasoline below 25 days or distillate below 30 days has historically preceded price spikes.

In [ ]:
fig = charts.plot_days_of_supply(df_dos)
fig.show()

### SPR Status

The Strategic Petroleum Reserve is the government's buffer against supply shocks. Drawdowns since 2022 have reduced this buffer significantly.

> **Reading this chart:** Lower SPR = less capacity to intervene during supply disruptions. When SPR share drops below 40%, the effective supply cushion is thin.

In [ ]:
fig = charts.plot_spr_status(df_spr)
fig.show()

---
## Futures Divergence — Physical vs Market

Compare commodity futures price z-scores to the physical delivery gap z-score. When physical supply is tighter than the market is pricing, there's a potential long entry.

> **Reading this chart:** When the physical composite gap score (from the scorecard above) is more negative than the futures z-score, the market may be underpricing supply risk.

In [ ]:
from commodity_flow import futures

try:
    df_fut = futures.fetch_futures_curves()
    df_fut = futures.compute_futures_z_scores(df_fut)
    if not df_fut.empty:
        fig = charts.plot_futures_divergence(df_fut)
        fig.show()
    else:
        print('Futures data unavailable.')
except Exception as e:
    print(f'Futures data unavailable: {e}')

---
## Helium Supply Chain Risk

Helium has no substitute. It is extracted as a byproduct of natural gas processing (cryogenic separation). Any disruption to NatGas throughput cascades directly to helium availability.

**Units:**
- **Mcm** — million cubic meters (helium production/demand volume)
- **Mcf** — thousand cubic feet (helium pricing, 1 Mcm ≈ 35.3 MMcf)
- **Bcf** — billion cubic feet (natural gas import volume)

In [ ]:
# Helium supply-demand chart
fig = charts.plot_helium_supply(data['helium'])
fig.show()

df_helium = data['helium']
latest_he = df_helium.iloc[-1]

# NatGas status
df_ng = data['natgas']
pipeline_avg = df_ng[df_ng['mode'] == 'Pipeline']['value_bcf'].tail(6).mean()
pipeline_ma12 = df_ng[df_ng['mode'] == 'Pipeline']['value_bcf'].mean()
ng_status = 'DISRUPTED' if pipeline_avg < pipeline_ma12 * 0.9 else 'STABLE'

print(f'NatGas Pipeline Imports (6mo avg): {pipeline_avg:.0f} Bcf/mo [{ng_status}]')
print(f'Helium Supply Gap: {latest_he["supply_gap_Mcm"]:+.0f} Mcm (world production - demand)')
print(f'BLM Grade-A Price: ${latest_he["blm_price_usd_per_mcf"]:.0f}/Mcf')
print()
if latest_he['supply_gap_Mcm'] < 0:
    print('\u26a0\ufe0f Helium market is in DEFICIT. Demand exceeds production by '
          f'{abs(latest_he["supply_gap_Mcm"]):.0f} Mcm. '
          'Affected industries: MRI systems, semiconductor fab, quantum computing, fiber optics.')
else:
    print('\u2705 Helium market balanced. Monitor NatGas throughput for leading indicator changes.')

---
## What to Watch

Key thresholds and upcoming catalysts.

In [ ]:
actual_sc = scorecard[~scorecard['is_forecast']]
latest_composite = actual_sc['composite_gap_score'].iloc[-1] if not actual_sc.empty else 0

print('=== Thresholds That Would Escalate Signals ===')
print(f'  Composite gap below -2.0\u03c3 (currently {latest_composite:.1f}\u03c3) \u2192 CRITICAL alert')

# Find WTI where 50% at risk
fifty_pct = risk_curve[risk_curve['pct_at_risk'] >= 50]
if not fifty_pct.empty:
    wti_50 = fifty_pct.iloc[0]['wti_price']
    print(f'  WTI below ${wti_50:.0f}/bbl \u2192 50% of US production at risk')

n_at_risk = (status['status'] == 'at risk').sum()
print(f'  3+ basins below breakeven (currently {n_at_risk}) \u2192 systemic shut-in risk')

# STEO forward
forecast_sc = scorecard[scorecard['is_forecast']]
if not forecast_sc.empty:
    fwd_avg = forecast_sc['composite_gap_score'].mean()
    print(f'\n=== STEO Forward View ===')
    print(f'  STEO-implied forward gap score: {fwd_avg:.1f}\u03c3 (avg over forecast horizon)')
    if fwd_avg < -1:
        print('  \u26a0\ufe0f EIA projections imply continued supply tightness.')
    else:
        print('  \u2705 EIA projections imply supply normalization.')

---
## Trade Ideas

*Analytical signals, not investment advice. All positions require independent due diligence.*

In [ ]:
ideas = []

# Double signal: gap + shut-in
if latest_composite < -1 and n_at_risk >= 2:
    ideas.append({
        'Signal': 'Double Signal: Gap + Shut-in',
        'Reading': f'Gap {latest_composite:.1f}\u03c3, {n_at_risk} basins at risk',
        'Direction': 'Long',
        'Instruments': 'WTI calendar spreads (backwardation steepening), Brent/WTI basis',
        'Rationale': 'Physical supply contraction accelerating faster than market expects',
    })

# Distillate tightness
if not df_dos.empty:
    dist_dos = df_dos[df_dos['product'] == 'EPD0']['days_of_supply']
    if not dist_dos.empty and dist_dos.iloc[-1] < 30:
        ideas.append({
            'Signal': 'Distillate Inventory Tight',
            'Reading': f'{dist_dos.iloc[-1]:.0f} days of supply (threshold: 30)',
            'Direction': 'Long',
            'Instruments': 'Heating oil futures (HO=F), diesel crack spreads',
            'Rationale': 'Physical distillate stocks below seasonal norm',
        })

# Helium deficit
if latest_he['supply_gap_Mcm'] < -10:
    ideas.append({
        'Signal': 'Helium Supply Deficit',
        'Reading': f'{latest_he["supply_gap_Mcm"]:+.0f} Mcm gap',
        'Direction': 'Long',
        'Instruments': 'Helium equities (DME, RLPI, PHX, AP), procurement contracts',
        'Rationale': 'No substitute; deficit growing; NatGas throughput is the supply bottleneck',
    })

# SPR low
if not df_spr.empty:
    spr_latest = df_spr.dropna().iloc[-1]['spr_pct']
    if spr_latest < 45:
        ideas.append({
            'Signal': 'SPR Buffer Depleted',
            'Reading': f'SPR at {spr_latest:.0f}% of total crude stocks',
            'Direction': 'Long tail risk',
            'Instruments': 'OTM WTI calls, energy vol (OVX)',
            'Rationale': 'Reduced government capacity to buffer supply shocks increases tail risk',
        })

if ideas:
    df_ideas = pd.DataFrame(ideas)
    display(df_ideas.style.set_properties(**{
        'text-align': 'left',
        'font-size': '12px',
        'white-space': 'pre-wrap',
    }).hide(axis='index'))
else:
    print('No signals currently firing at actionable levels.')
    print('Monitor the signal status table above for threshold breaches.')

---
## Seasonal Decomposition — Petroleum Demand Patterns

STL (Seasonal-Trend-Loess) decomposition separates the underlying trend from seasonal patterns and residual shocks. Estimated on the post-2017 structural regime — after the shale revolution stabilized US crude imports at their refinery-configuration floor and the crude export ban was lifted.

> **Reading this chart:** The **Trend** panel shows the structural direction of imports stripped of seasonality. The **Seasonal** panel shows the repeating annual pattern (winter heating demand, summer driving season). The **Residual** panel shows unexplained shocks — red dots mark anomalies exceeding 2σ. These are the supply disruptions the market should be watching.

In [ ]:
# Fetch extended import history for stable seasonal estimation
# STL needs 5+ annual cycles. Start from 2017 (current structural
# regime: post-shale ramp, post-export ban, imports at refinery floor)
if config.USE_LIVE_API and config.EIA_API_KEY:
    from commodity_flow import eia as _eia
    _df_extended = _eia.fetch_crude_imports_by_padd(config.EIA_API_KEY, start='2017-01')
else:
    _df_extended = data['imports']  # offline: 2022-present (limited)

national_imports = _df_extended.groupby('date')['value'].sum().sort_index()
national_imports.index = pd.DatetimeIndex(national_imports.index, freq='MS')

if len(national_imports) >= 36:
    fig = charts.plot_seasonal_decomposition(
        national_imports, period=12, display_years=3,
        title='Crude Oil Imports — Seasonal Decomposition (STL)'
    )
    fig.show()
else:
    print(f'Insufficient data for seasonal decomposition ({len(national_imports)} months, need 36+).')


---
## Data Provenance

In [ ]:
from IPython.display import Markdown
display(Markdown(pipeline.provenance.summary()))